# Importing Libraries

In [37]:
!pip install -q transformers datasets accelerate evaluate scikit-learn optuna

In [38]:
import numpy as np
import pandas as pd
import torch
import random
import optuna
import evaluate

from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)
from datasets import Dataset
from tqdm.auto import tqdm
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

In [39]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# Importing datasets

In [40]:
! gdown 1o_E9XCDt0iMkcJhvwQbRrVpfo3FAi2nq
! gdown 1U_NPI_FyBOMSNtcJEBNONDeYoOLMNIrY
! gdown 11uurHsNZAjG70Ooq0noI7a2W4rbiNvIw

Downloading...
From: https://drive.google.com/uc?id=1o_E9XCDt0iMkcJhvwQbRrVpfo3FAi2nq
To: /content/jigsaw_bert_dataset.csv
100% 20.9M/20.9M [00:00<00:00, 63.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1U_NPI_FyBOMSNtcJEBNONDeYoOLMNIrY
To: /content/twitter_bert_dataset.csv
100% 1.15M/1.15M [00:00<00:00, 12.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=11uurHsNZAjG70Ooq0noI7a2W4rbiNvIw
To: /content/combined_model_dataset.csv
100% 22.0M/22.0M [00:00<00:00, 93.1MB/s]


In [41]:
df_google = pd.read_csv('/content/jigsaw_bert_dataset.csv')
df_twitter = pd.read_csv('/content/twitter_bert_dataset.csv')
df_combined = pd.read_csv('/content/combined_model_dataset.csv')


# Data preperation

## Text preprocessing

In [42]:
!pip install contractions

In [43]:
import re
import nltk
import contractions
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))- {"not", "no"}

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [44]:
def preprocess_bert(text):
  if not text or str(text).strip() == "":
    return ""
  # lowercase
  text = text.lower()

  # remove mentions
  text = re.sub(r'@\w+', 'user', text)

  # remove hashtag symbol but keep word
  text = re.sub(r'#', '', text)

  # remove URLs
  text = re.sub(r'http\S+|www\S+', '', text)

  # remove black bars
  text = re.sub(r'█+', ' ', text)

  # remove extra spaces
  text = re.sub(r'\s+', ' ', text).strip()

  return text

In [45]:
df_google["text"] = df_google["text"].apply(preprocess_bert)
df_google

,text,label
0,"was the girl on any prescription drugs? if so,...",0
1,"while the tax will have an effect, the real ef...",0
2,what makes you think i believe the posts in qu...,0
3,and so canada devolves into nonfunctional irre...,0
4,"boomer 88, a rather defeatist attitude and lam...",0
...,...,...
69995,and another childish insult from clearly an em...,1
69996,"no, you just call them deviants and perverts.",1
69997,"well said, bumpkin. i'm more inclined to liste...",1
69998,did i mention that mars is undergoing climate ...,1


In [46]:
df_twitter["text"] = df_twitter["text"].apply(preprocess_bert)
df_twitter

,text,label
0,user nice new signage. are you not concerned b...,0
1,a woman who you fucked multiple times saying y...,1
2,user user real talk do you have eyes or were t...,1
3,your girlfriend lookin at me like a groupie in...,1
4,hysterical woman like user,0
...,...,...
8949,oooohhhh bitch didn't even listen to the dead ...,0
8950,user good luck user more americans walkawayfro...,0
8951,bitch you can't keep up so stop trying,1
8952,user user user user user user japan is always ...,0


In [47]:
df_combined["text"] = df_combined["text"].apply(preprocess_bert)
df_combined

,text,label
0,"this is politicians' strategy ""action through ...",0
1,he has underlings to do tjat for him,0
2,"judge on separation of immigrant families: ""if...",0
3,to the republican nra(nazi's razing america)th...,0
4,somebody musta complained.,0
...,...,...
78949,throw a little pesticide on top of carcinogen ...,0
78950,this man is a coward & an oath breaker & the w...,1
78951,a propublica investigation found that police r...,0
78952,you answered the question better than i could ...,0


## Spliting dataset into train and test

## Splitting combined dataset

In [48]:
X = df_combined['text']
y = df_combined['label']

In [49]:
X_tr, X_test, y_tr, y_test = train_test_split(X,y,test_size = 0.20, random_state = 42, stratify = y)
X_train, X_val, y_train, y_val = train_test_split(X_tr,y_tr,test_size = 0.25, random_state = 42, stratify = y_tr)

In [50]:
X_train.shape

(47372,)

In [51]:
X_test.shape

(15791,)

## Splitting google, twitter and reddit datasets

In [52]:
google_train_df,google_val_df = train_test_split(
    df_google,
    test_size = 0.2,
    random_state = SEED,
    stratify = df_google['label']
)

In [53]:
twitter_train_df, twitter_temp_df = train_test_split(
    df_twitter,
    test_size = 0.4,
    random_state = SEED,
    stratify = df_twitter['label']
)

twitter_val_df, twitter_test_df = train_test_split(
    twitter_temp_df,
    test_size = 0.5,
    random_state = SEED,
    stratify = twitter_temp_df['label']
)

In [54]:
print("Google train:", google_train_df.shape)
print("Google val:", google_val_df.shape)
print("Twitter train:", twitter_train_df.shape)
print("Twitter val:", twitter_val_df.shape)
print("Twitter test:", twitter_test_df.shape)

Google train: (56000, 2)
Google val: (14000, 2)
Twitter train: (5372, 2)
Twitter val: (1791, 2)
Twitter test: (1791, 2)


##  Converting to Hugging Face datasets

In [55]:
google_train_ds = Dataset.from_pandas(google_train_df.reset_index(drop=True))
google_val_ds = Dataset.from_pandas(google_val_df.reset_index(drop=True))
google_train_ds

Dataset({
    features: ['text', 'label'],
    num_rows: 56000
})

In [56]:
twitter_train_ds = Dataset.from_pandas(twitter_train_df.reset_index(drop=True))
twitter_val_ds = Dataset.from_pandas(twitter_val_df.reset_index(drop=True))
twitter_test_ds = Dataset.from_pandas(twitter_test_df.reset_index(drop=True))
twitter_train_ds

Dataset({
    features: ['text', 'label'],
    num_rows: 5372
})

## Tokenization

In [57]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [58]:
MAX_LENGTH = 128
def tokenize_batch(batch):
  return tokenizer(
      batch['text'],
      truncation = True,
      max_length = MAX_LENGTH
  )

In [59]:
google_train_tok = google_train_ds.map(tokenize_batch, batched = True)
google_val_tok = google_val_ds.map(tokenize_batch, batched = True)


In [60]:
twitter_train_tok = twitter_train_ds.map(tokenize_batch, batched = True)
twitter_val_tok = twitter_val_ds.map(tokenize_batch, batched = True)
twitter_test_tok = twitter_test_ds.map(tokenize_batch, batched = True)


In [61]:
twitter_test_tok

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1791
})

## Set dataset format

In [62]:
cols = ['input_ids','attention_mask','label']
google_train_tok.set_format(type = 'torch', columns = cols)
google_val_tok.set_format(type = 'torch', columns = cols)

In [63]:
twitter_train_tok.set_format(type = 'torch', columns = cols)
twitter_val_tok.set_format(type = 'torch', columns = cols)
twitter_test_tok.set_format(type = 'torch', columns = cols)

# Evaluation metrics

In [64]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    prec = precision_metric.compute(predictions=preds, references=labels, average="binary")["precision"]
    rec = recall_metric.compute(predictions=preds, references=labels, average="binary")["recall"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }

# Hyperparameter tuning

In [65]:
def build_model():
  return AutoModelForSequenceClassification.from_pretrained(
      MODEL_NAME,
      num_labels = 2
  )
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

In [66]:
def objective(trial):
  learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
  batch_size = trial.suggest_categorical("batch_size",[8,16])
  num_train_epochs = trial.suggest_int("num_train_epochs",1,2)
  weight_decay = trial.suggest_float("weight_decay", 0.0, 1.0)
  model = build_model()
  args = TrainingArguments(
      output_dir = f"./stage1_trial_{trial.number}",
      eval_strategy = "epoch",
      save_strategy = 'epoch',
      logging_strategy = "epoch",
      learning_rate = learning_rate,
      per_device_train_batch_size = batch_size,
      per_device_eval_batch_size = batch_size,
      weight_decay = weight_decay,
      load_best_model_at_end = True,
      metric_for_best_model = 'f1',
      greater_is_better = True,
      report_to = "none",
      seed = SEED,
      fp16=torch.cuda.is_available()
  )
  trainer = Trainer(
      model = model,
      args = args,
      train_dataset = google_train_tok,
      eval_dataset = google_val_tok,
      data_collator = data_collator,
      compute_metrics = compute_metrics,
      callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]

  )
  trainer.train()
  eval_results = trainer.evaluate()
  return eval_results["eval_f1"]

In [68]:
study = optuna.create_study(direction = "maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials = 5)
print("Best params:", study.best_params)
print("Best F1:", study.best_value)

[I 2026-04-06 00:08:08,019] A new study created in memory with name: no-name-954392bf-be47-4fa6-9bd4-12bd23b123f1


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


[W 2026-04-06 00:08:10,316] Trial 0 failed with parameters: {'learning_rate': 1.827226177606625e-05, 'batch_size': 8, 'num_train_epochs': 2, 'weight_decay': 0.15601864044243652} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_8336/2557983413.py", line 33, in objective
    trainer.train()
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 2174, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 2480, in _inner_training_loop
    batch_samples, num_items_in_batch = self.get_batch_samples(epoch_iterator, num_batches, args.device)
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  F

KeyboardInterrupt: 